# Exercise 1.2: Supervised Learning with scikit-learn and PyTorch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/01_machine_learning/notebooks/exercise_1_2_supervised_learning.ipynb)

This notebook introduces a compact supervised learning workflow. It uses small synthetic datasets so that the model behavior can be inspected directly in tables, curves, and plots.

You will practice:

- linear regression with `scikit-learn`
- linear regression training with `torch` and gradient descent
- underfitting and overfitting with polynomial features
- logistic regression for binary classification
- basic regression and classification evaluation


## 0. Setup

Run this first in Colab. The datasets are generated inside the notebook; no external data download is needed.


In [ ]:
%pip -q install scikit-learn pandas matplotlib seaborn torch

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

import torch
from torch import nn

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)


## 1. A tiny regression problem

We create a housing-like example. The feature is apartment size. The target is monthly rent.

This is supervised learning because every training row has an input `X` and a known answer `y`.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

n = 80
living_area = rng.uniform(20, 120, size=n)
rent = 320 + 11.5 * living_area + rng.normal(0, 95, size=n)

regression_df = pd.DataFrame({"living_area_m2": living_area, "monthly_rent_eur": rent})
display(regression_df.head())

fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(data=regression_df, x="living_area_m2", y="monthly_rent_eur", ax=ax)
ax.set_title("Small regression dataset")
plt.show()


## 2. Linear regression with scikit-learn

The workflow is:

1. split the data into train and test sets
2. fit the model on the train set
3. predict on the test set
4. evaluate with metrics
5. draw the fitted line on top of the data


In [ ]:
X = regression_df[["living_area_m2"]]
y = regression_df["monthly_rent_eur"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
y_pred = linear_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Intercept: {linear_model.intercept_:.1f}")
print(f"Slope:     {linear_model.coef_[0]:.2f} EUR per m2")
print(f"MAE:       {mae:.1f} EUR")
print(f"RMSE:      {rmse:.1f} EUR")
print(f"R2:        {r2:.3f}")

x_line = np.linspace(X["living_area_m2"].min(), X["living_area_m2"].max(), 100).reshape(-1, 1)
y_line = linear_model.predict(x_line)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(X_train, y_train, label="train", alpha=0.75)
ax.scatter(X_test, y_test, label="test", alpha=0.9)
ax.plot(x_line, y_line, color="black", linewidth=2.5, label="fitted linear model")
ax.set_xlabel("Living area (m2)")
ax.set_ylabel("Monthly rent (EUR)")
ax.set_title("Linear regression fit shown on the data")
ax.legend()
plt.show()


### Student task 1: Predict new values

Use the template below to predict rent for several apartment sizes. Try values inside the training range and outside the training range.


In [ ]:
# Student template: change these values and rerun the cell.
new_areas = pd.DataFrame({"living_area_m2": [35, 65, 95, 140]})
new_areas["predicted_rent_eur"] = linear_model.predict(new_areas)
display(new_areas)


<details>
<summary>Answer / tip</summary>

Predictions inside the observed range are usually more reliable. Predictions far outside the observed range are extrapolations: the line continues, but the model has not seen comparable apartments.

A useful check is to compare the new values with `regression_df["living_area_m2"].min()` and `.max()`.
</details>


In [ ]:
# Optional check after Student task 1.
print("Observed size range:")
print(regression_df["living_area_m2"].agg(["min", "max"]))

task1_check = pd.DataFrame({"living_area_m2": [25, 50, 100, 150]})
task1_check["predicted_rent_eur"] = linear_model.predict(task1_check)
display(task1_check)


## 3. Watching gradient descent with PyTorch

`LinearRegression()` solves the line directly. In this section we train a linear regression model with PyTorch.

The model is still linear, but the parameters are learned step by step:

1. start with current weight and bias
2. predict rents
3. compute mean squared error
4. use backpropagation to compute gradients
5. update parameters with the optimizer

The printout every 5 epochs shows how the weight, bias, and MSE change during optimization.


In [ ]:
x_train_np = X_train[["living_area_m2"]].to_numpy(dtype="float32")
y_train_np = y_train.to_numpy(dtype="float32").reshape(-1, 1)
x_test_np = X_test[["living_area_m2"]].to_numpy(dtype="float32")
y_test_np = y_test.to_numpy(dtype="float32").reshape(-1, 1)

x_mean, x_std = x_train_np.mean(), x_train_np.std()
y_mean, y_std = y_train_np.mean(), y_train_np.std()

x_train_scaled = (x_train_np - x_mean) / x_std
y_train_scaled = (y_train_np - y_mean) / y_std
x_test_scaled = (x_test_np - x_mean) / x_std

X_train_t = torch.tensor(x_train_scaled)
y_train_t = torch.tensor(y_train_scaled)
X_test_t = torch.tensor(x_test_scaled)


def train_torch_linear(lr=0.08, epochs=100, print_every=5):
    torch.manual_seed(RANDOM_STATE)
    model = nn.Linear(1, 1)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    rows = []

    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        y_hat_scaled = model(X_train_t)
        loss = loss_fn(y_hat_scaled, y_train_t)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            train_pred_eur = model(X_train_t).numpy() * y_std + y_mean
            mse_eur = mean_squared_error(y_train_np, train_pred_eur)
            weight = model.weight.item()
            bias = model.bias.item()

        rows.append(
            {
                "epoch": epoch,
                "weight_scaled": weight,
                "bias_scaled": bias,
                "scaled_MSE_loss": loss.item(),
                "train_MSE_EUR2": mse_eur,
            }
        )

        if epoch == 1 or epoch % print_every == 0:
            print(
                f"epoch {epoch:03d} | "
                f"w={weight:+.3f} | b={bias:+.3f} | "
                f"scaled MSE={loss.item():.4f} | train MSE EUR2={mse_eur:.1f}"
            )

    return model, pd.DataFrame(rows)

torch_linear, gd_history = train_torch_linear(lr=0.08, epochs=100, print_every=5)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(gd_history["epoch"], gd_history["scaled_MSE_loss"])
axes[0].set_title("PyTorch linear regression loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Scaled MSE")

with torch.no_grad():
    torch_line_scaled = (x_line.astype("float32") - x_mean) / x_std
    torch_line_pred = torch_linear(torch.tensor(torch_line_scaled)).numpy() * y_std + y_mean

axes[1].scatter(X_train, y_train, label="train", alpha=0.75)
axes[1].scatter(X_test, y_test, label="test", alpha=0.9)
axes[1].plot(x_line, torch_line_pred, color="black", linewidth=2.5, label="PyTorch trained line")
axes[1].set_title("Line learned by gradient descent")
axes[1].set_xlabel("Living area (m2)")
axes[1].set_ylabel("Monthly rent (EUR)")
axes[1].legend()

plt.tight_layout()
plt.show()

display(gd_history.tail())


### Student task 2: Compare training curves

Run the template below and compare several learning rates. Look at both the curve shape and the final loss.


In [ ]:
# Student template: add or remove learning rates.
learning_rates = [0.005, 0.02, 0.08, 0.3]
curve_rows = []

for lr in learning_rates:
    _, hist = train_torch_linear(lr=lr, epochs=100, print_every=200)
    hist["learning_rate"] = lr
    curve_rows.append(hist)

lr_comparison = pd.concat(curve_rows, ignore_index=True)

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(data=lr_comparison, x="epoch", y="scaled_MSE_loss", hue="learning_rate", ax=ax)
ax.set_title("Gradient descent curves for different learning rates")
ax.set_ylabel("Scaled MSE")
plt.show()

summary = lr_comparison.groupby("learning_rate").tail(1)[
    ["learning_rate", "scaled_MSE_loss", "train_MSE_EUR2"]
]
display(summary)


<details>
<summary>Answer / tip</summary>

A very small learning rate improves slowly. A moderate learning rate usually reaches a low loss quickly. A very large learning rate can oscillate or diverge.

For this dataset, `0.08` should usually be a stable choice. `0.005` is slower. If a curve increases strongly or becomes unstable, the learning rate is too large.
</details>


## 4. Underfitting and overfitting

Now we use a small noisy curve. The data are deliberately limited to the range `[-1, 1]` so that high-degree polynomial models do not explode outside the observed range.

- Degree 1 is too simple and underfits.
- A moderate degree can capture the main curve.
- A high degree can start to wiggle between points and overfit noise.


In [ ]:
rng = np.random.default_rng(7)
x = np.sort(rng.uniform(-1.0, 1.0, size=26))
true_curve = lambda z: np.sin(2.6 * np.pi * z) + 0.55 * z
y_curve = true_curve(x) + rng.normal(0, 0.22, size=x.size)

X_curve = pd.DataFrame({"x": x})
y_curve_series = pd.Series(y_curve, name="target")

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_curve, y_curve_series, test_size=0.38, random_state=RANDOM_STATE
)


def fit_and_plot_polynomials(degrees):
    x_grid = np.linspace(X_curve["x"].min(), X_curve["x"].max(), 400).reshape(-1, 1)
    y_margin = 0.7
    y_min = y_curve_series.min() - y_margin
    y_max = y_curve_series.max() + y_margin

    results = []
    fig, axes = plt.subplots(1, len(degrees), figsize=(5 * len(degrees), 4), sharey=True)
    if len(degrees) == 1:
        axes = [axes]

    for ax, degree in zip(axes, degrees):
        model = make_pipeline(PolynomialFeatures(degree=degree, include_bias=False), LinearRegression())
        model.fit(Xc_train, yc_train)

        train_pred = model.predict(Xc_train)
        test_pred = model.predict(Xc_test)
        train_rmse = np.sqrt(mean_squared_error(yc_train, train_pred))
        test_rmse = np.sqrt(mean_squared_error(yc_test, test_pred))
        results.append({"degree": degree, "train_RMSE": train_rmse, "test_RMSE": test_rmse})

        ax.scatter(Xc_train["x"], yc_train, label="train", alpha=0.75)
        ax.scatter(Xc_test["x"], yc_test, label="test", alpha=0.9)
        ax.plot(x_grid.ravel(), model.predict(x_grid), color="black", linewidth=2)
        ax.set_ylim(y_min, y_max)
        ax.set_title(f"Polynomial degree {degree}")
        ax.set_xlabel("x")
        ax.set_ylabel("target")

    axes[0].legend()
    plt.tight_layout()
    plt.show()
    return pd.DataFrame(results)

underfit_results = fit_and_plot_polynomials([1, 5, 10])
display(underfit_results)


### Student task 3: Try other polynomial degrees

Use the template below. Compare the pictures and the test RMSE.


In [ ]:
# Student template: try your own degree list.
degrees_to_try = [1, 2, 3, 5, 7, 10]
student_degree_results = fit_and_plot_polynomials(degrees_to_try)
display(student_degree_results.sort_values("test_RMSE"))


<details>
<summary>Answer / tip</summary>

A low degree usually underfits: both train and test errors are high, and the curve misses the pattern. A very high degree often has lower train error but can wiggle between points and give worse test error.

The exact best degree can change with the random split, but a moderate degree should usually be more plausible than degree 1 or degree 10. If the high-degree curve becomes visually unstable, that is already a useful sign of overfitting.
</details>


## 5. Logistic regression for classification

The target is now a class. We create a two-feature dataset for bike-sharing demand: `near_transit_score` and `job_density_score`.

The two classes overlap slightly, but logistic regression should reach about 90 percent accuracy on the test set.


In [ ]:
rng = np.random.default_rng(21)
n_low = 95
n_high = 95

low = rng.multivariate_normal(mean=[-1.1, -0.9], cov=[[0.55, 0.12], [0.12, 0.55]], size=n_low)
high = rng.multivariate_normal(mean=[1.2, 1.0], cov=[[0.65, 0.18], [0.18, 0.60]], size=n_high)

X_cls_np = np.vstack([low, high])
y_cls_np = np.array([0] * n_low + [1] * n_high)

classification_df = pd.DataFrame(
    X_cls_np,
    columns=["near_transit_score", "job_density_score"],
)
classification_df["high_demand"] = y_cls_np
classification_df["label"] = np.where(y_cls_np == 1, "high demand", "low demand")

display(classification_df.head())

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.scatterplot(
    data=classification_df,
    x="near_transit_score",
    y="job_density_score",
    hue="label",
    palette=["#4C78A8", "#F58518"],
    s=55,
    edgecolor="white",
    linewidth=0.5,
    ax=ax,
)
ax.set_title("Bike-sharing demand classification dataset")
plt.show()


In [ ]:
X_cls = classification_df[["near_transit_score", "job_density_score"]]
y_cls = classification_df["high_demand"]

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.30, random_state=RANDOM_STATE, stratify=y_cls
)

logistic_model = make_pipeline(StandardScaler(), LogisticRegression())
logistic_model.fit(X_train_cls, y_train_cls)

y_pred_cls = logistic_model.predict(X_test_cls)
y_prob_cls = logistic_model.predict_proba(X_test_cls)[:, 1]

print(f"Accuracy: {accuracy_score(y_test_cls, y_pred_cls):.3f}")
print(classification_report(y_test_cls, y_pred_cls, target_names=["low demand", "high demand"]))


In [ ]:
def plot_logistic_boundary(model, X, y, title):
    x_min, x_max = X.iloc[:, 0].min() - 0.6, X.iloc[:, 0].max() + 0.6
    y_min, y_max = X.iloc[:, 1].min() - 0.6, X.iloc[:, 1].max() + 0.6
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
    grid = pd.DataFrame(
        {"near_transit_score": xx.ravel(), "job_density_score": yy.ravel()}
    )
    zz = model.predict_proba(grid)[:, 1].reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(7, 5))
    contour = ax.contourf(xx, yy, zz, levels=np.linspace(0, 1, 11), cmap="RdYlBu_r", alpha=0.35)
    ax.contour(xx, yy, zz, levels=[0.5], colors="black", linewidths=2)
    sns.scatterplot(
        x=X.iloc[:, 0],
        y=X.iloc[:, 1],
        hue=np.where(y == 1, "high demand", "low demand"),
        palette=["#4C78A8", "#F58518"],
        s=55,
        edgecolor="white",
        linewidth=0.5,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("near_transit_score")
    ax.set_ylabel("job_density_score")
    fig.colorbar(contour, ax=ax, label="P(high demand)")
    plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test_cls, y_pred_cls, display_labels=["low", "high"], ax=axes[0], colorbar=False
)
axes[0].set_title("Confusion matrix")
RocCurveDisplay.from_predictions(y_test_cls, y_prob_cls, ax=axes[1])
axes[1].set_title("ROC curve")
plt.tight_layout()
plt.show()

plot_logistic_boundary(logistic_model, X_test_cls, y_test_cls, "Logistic regression decision boundary")


### Student task 4: Change the classification threshold

The default threshold is `0.5`. A lower threshold predicts more high-demand cases. A higher threshold predicts fewer.


In [ ]:
# Student template: change threshold and compare the confusion matrix.
threshold = 0.35
custom_pred = (y_prob_cls >= threshold).astype(int)

print(f"Threshold: {threshold}")
print(f"Accuracy:  {accuracy_score(y_test_cls, custom_pred):.3f}")
print(confusion_matrix(y_test_cls, custom_pred))

ConfusionMatrixDisplay.from_predictions(
    y_test_cls,
    custom_pred,
    display_labels=["low", "high"],
    colorbar=False,
)
plt.title(f"Confusion matrix with threshold {threshold}")
plt.show()


<details>
<summary>Answer / tip</summary>

Lowering the threshold usually increases true positives for high demand, but it can also increase false positives. Raising the threshold does the opposite.

Use a low threshold if missing high-demand cases is costly. Use a high threshold if false alarms are costly.
</details>


## 6. Logistic regression training with PyTorch

This repeats the binary classification idea in PyTorch. The model is one linear layer with one output logit.

`BCEWithLogitsLoss` combines sigmoid activation and binary cross-entropy in a numerically stable way.


In [ ]:
cls_scaler = StandardScaler()
X_train_cls_scaled = cls_scaler.fit_transform(X_train_cls).astype("float32")
X_test_cls_scaled = cls_scaler.transform(X_test_cls).astype("float32")

X_train_log_t = torch.tensor(X_train_cls_scaled)
y_train_log_t = torch.tensor(y_train_cls.to_numpy().reshape(-1, 1).astype("float32"))
X_test_log_t = torch.tensor(X_test_cls_scaled)
y_test_log_np = y_test_cls.to_numpy().reshape(-1, 1)


def train_torch_logistic(lr=0.12, epochs=300, print_every=25):
    torch.manual_seed(RANDOM_STATE)
    model = nn.Linear(2, 1)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    rows = []

    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        logits = model(X_train_log_t)
        loss = loss_fn(logits, y_train_log_t)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            train_prob = torch.sigmoid(model(X_train_log_t)).numpy()
            test_prob = torch.sigmoid(model(X_test_log_t)).numpy()
        train_acc = ((train_prob >= 0.5).astype(int) == y_train_log_t.numpy()).mean()
        test_acc = ((test_prob >= 0.5).astype(int) == y_test_log_np).mean()
        rows.append({"epoch": epoch, "loss": loss.item(), "train_accuracy": train_acc, "test_accuracy": test_acc})

        if epoch == 1 or epoch % print_every == 0 or epoch == epochs:
            print(
                f"epoch {epoch:03d} | loss={loss.item():.4f} | "
                f"train acc={train_acc:.3f} | test acc={test_acc:.3f}"
            )

    history = pd.DataFrame(rows)
    final = history.iloc[-1]
    print(
        f"Final after {epochs} epochs: "
        f"loss={final['loss']:.4f}, "
        f"train acc={final['train_accuracy']:.3f}, "
        f"test acc={final['test_accuracy']:.3f}"
    )
    return model, history

torch_logistic, logistic_history = train_torch_logistic(lr=0.12, epochs=300, print_every=25)

display(logistic_history.tail())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(logistic_history["epoch"], logistic_history["loss"])
axes[0].set_title("PyTorch logistic regression loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE loss")

axes[1].plot(logistic_history["epoch"], logistic_history["train_accuracy"], label="train")
axes[1].plot(logistic_history["epoch"], logistic_history["test_accuracy"], label="test")
axes[1].set_title("Accuracy during training")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1.02)
axes[1].legend()
plt.tight_layout()
plt.show()


### Student task 5: Compare PyTorch logistic training curves

Run the template below with several learning rates. The training is intentionally longer here, so slower learning rates have enough time to approach a low loss.

Compare the curve shape and the final table.


In [ ]:
# Student template: compare multiple learning rates.
logistic_lrs = [0.01, 0.05, 0.12, 0.5]
comparison_epochs = 300
all_histories = []

for lr in logistic_lrs:
    _, hist = train_torch_logistic(lr=lr, epochs=comparison_epochs, print_every=comparison_epochs)
    hist["learning_rate"] = lr
    all_histories.append(hist)

logistic_lr_df = pd.concat(all_histories, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.lineplot(data=logistic_lr_df, x="epoch", y="loss", hue="learning_rate", ax=axes[0])
axes[0].set_title("PyTorch logistic loss for different learning rates")
axes[0].set_ylabel("BCE loss")

sns.lineplot(data=logistic_lr_df, x="epoch", y="test_accuracy", hue="learning_rate", ax=axes[1], legend=False)
axes[1].set_title("Test accuracy for different learning rates")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1.02)

plt.tight_layout()
plt.show()

final_summary = (
    logistic_lr_df.sort_values("epoch")
    .groupby("learning_rate")
    .tail(1)
    [["learning_rate", "epoch", "loss", "train_accuracy", "test_accuracy"]]
    .rename(
        columns={
            "loss": "final_loss",
            "train_accuracy": "final_train_accuracy",
            "test_accuracy": "final_test_accuracy",
        }
    )
    .sort_values("final_loss")
)

display(final_summary)
print(f"Lowest final loss: {final_summary['final_loss'].min():.4f}")


<details>
<summary>Answer / tip</summary>

With more epochs, the small learning rates should continue decreasing instead of stopping early. A larger learning rate often reaches a lower loss faster, but if it is too large the loss can become unstable.

Use the final table to compare models numerically. In this example, `final_loss` tells you how low the binary cross-entropy became after the same number of epochs.
</details>


## 7. Short evaluation checklist

Use these questions when you train a supervised model:

1. What is the target variable? Is it continuous or categorical?
2. What features are available at prediction time?
3. Did you keep a test set separate from training?
4. For regression: are MAE, RMSE, and R2 acceptable for the use case?
5. For classification: what errors are shown by the confusion matrix?
6. Does the model underfit, overfit, or generalize reasonably?
7. Could nearby spatial samples cause leakage between train and test data?
